AutoRAG Banner

## Notebook content

This notebook presents the AutoRAG steps: data preparation, experiment execution, leaderboard analysis of the generated RAG patterns, and querying the selected pattern.

### Contents 
This notebook contains the following parts:
- **[Setup](#Setup)**
- **[Prepare experiment data](#Prepare-experiment-data)**
- **[Process input documents](#Process-input-documents)**
- **[Run ai4rag experiment](#Run-ai4rag-experiment)**
- <span style="color:red">TODO</span>
- **[Summary and next steps](#Summary-and-next-steps)**

## Setup

Install packages

In [1]:
import sys
!{sys.executable} -m pip install boto3 --no-cache-dir git+https://github.com/LukaszCmielowski/pipelines-components.git@rhoai_autorag_data_processing_pipeline 2>&1 | tail -n 10

  Created wheel for kfp-components: filename=kfp_components-1.11.0-py3-none-any.whl size=26895 sha256=0281c88fa86486693e4239e37aa26bcc1c180b2daea8d3b8a0a6083ae492493d
  Stored in directory: /private/var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/pip-ephem-wheel-cache-iac3d_2_/wheels/62/7f/40/c65ae1bd9a71463506539fcf88d59a7aa6927696e1d20dc2cb
Successfully built kfp-components

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Import required libraries

In [2]:
import re
import os
import json
import yaml
import logging
import urllib.request
from pathlib import Path
from types import SimpleNamespace

import warnings
warnings.filterwarnings("ignore")

import ibm_boto3
from ibm_botocore.client import Config
from langchain_core.documents import Document

for logger_name in (
    "Test Data Loader component logger",
    "Document Loader component logger",
    "Text Extraction component logger",
):
    logging.getLogger(logger_name).propagate = False

📌 **Action**: Provide the credentials for your S3 instance if they are not already set in the notebook environment.

In [4]:
AWS_ACCESS_KEY_ID = ""
AWS_SECRET_ACCESS_KEY = ""
AWS_S3_ENDPOINT = ""
AWS_DEFAULT_REGION = ""

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID or os.environ.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY or os.environ.get("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_S3_ENDPOINT"] = AWS_S3_ENDPOINT or os.environ.get("AWS_S3_ENDPOINT")
os.environ["AWS_DEFAULT_REGION"] = AWS_DEFAULT_REGION or os.environ.get("AWS_DEFAULT_REGION")

📌 **Action**: Provide the bucket name where the experiment data will be stored.

> 🔖 **Note**: Bucket must already exists.

In [5]:
BUCKET_NAME = ""

## Prepare experiment data

#### Initialize S3 client

In [6]:
s3_client = ibm_boto3.client(
    "s3",
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
    config=Config(signature_version="s3v4"),
)

#### Upload documents
For the needs of this notebook we are using IBM financial reports available here: https://www.ibm.com/investor/financial-reporting

In [8]:
documents_urls = [
    "https://www.ibm.com/downloads/documents/us-en/12bb2f913a3ba1a2",
    "https://www.ibm.com/downloads/documents/us-en/131cf8a39db327fd",
    "https://www.ibm.com/downloads/documents/us-en/131cf87ab633199f",
    "https://www.ibm.com/downloads/documents/us-en/1550f7eea8c0ded6",
    "https://www.ibm.com/downloads/documents/us-en/10a9980400afd114",
    "https://www.ibm.com/downloads/documents/us-en/10a9980468afdf4c",
    "https://www.ibm.com/downloads/documents/us-en/10a9980400afd11c",
    "https://www.ibm.com/downloads/documents/us-en/11ed3283ae56ec71"
]

for url in documents_urls:
    with urllib.request.urlopen(url) as response:
        content = response.read()
        content_disposition = response.headers.get("Content-Disposition")
        filename = re.findall('filename="(.+)"', content_disposition)[0]
        s3_client.put_object(Bucket=BUCKET_NAME, Key=filename, Body=content)
        print(filename)

ibm-1q25-earnings-press-release.pdf
ibm-2q25-earnings-press-release.pdf
ibm-3q-25-press-release.pdf
4q25-press-release.pdf
ibm-1q24-earnings-press-release.pdf
ibm-3q24-earnings-press-release.pdf
ibm-2q24-earnings-press-release.pdf
ibm-4q24-earnings-press-release.pdf


#### Upload benchmark dataset

In [9]:
benchmark = [
    {
        "question": "What was IBM's revenue in the first quarter of 2024?",
        "correct_answers": [
            "Revenue of $14.5 billion, up 1 percent, up 3 percent at constant currency."
        ],
        "correct_answer_document_ids": [
            "ibm-1q25-earnings-press-release.pdf"
        ]
    },
    {
        "question": "What did IBM announce regarding HashiCorp in first quarter 2024?",
        "correct_answers": [
            "IBM announced its intent to acquire HashiCorp, Inc. for $35 per share in cash, representing an enterprise value of $6.4 billion. The transaction was expected to close by the end of 2024."
        ],
        "correct_answer_document_ids": [
            "ibm-1q25-earnings-press-release.pdf"
        ]
    }
]

res = s3_client.put_object(Bucket=BUCKET_NAME, Key="benchmark.json", Body=json.dumps(benchmark))

Look up bucket contents

In [10]:
res = s3_client.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix="",
).get("Contents", [])

print("Bucket contents:")
for r in res:
    print(r["Key"])

Bucket contents:
4q25-press-release.pdf
benchmark.json
ibm-1q24-earnings-press-release.pdf
ibm-1q25-earnings-press-release.pdf
ibm-2q24-earnings-press-release.pdf
ibm-2q25-earnings-press-release.pdf
ibm-3q-25-press-release.pdf
ibm-3q24-earnings-press-release.pdf
ibm-4q24-earnings-press-release.pdf


## Process input documents

The data processing flow prepares input for the experiment in three steps. Each step runs as a standalone component (via `python_func`) with artifact paths under `step_outputs/`. Ensure `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_S3_ENDPOINT`, and `AWS_DEFAULT_REGION` are set in your environment before running.

| Step | Component | Purpose |
|------|-----------|---------|
| 1 | **Test data loader** | Download the benchmark JSON (questions and ground truth) from S3. |
| 2 | **Documents sampling** | List documents in the bucket, prioritize benchmark-referenced docs, apply a size cap, and write a YAML manifest (no content download). |
| 3 | **Text extraction** | Download the listed documents from S3 and extract text to Markdown using Docling. |

In [11]:
from kfp_components.components.data_processing.autorag.test_data_loader.component import test_data_loader
from kfp_components.components.data_processing.autorag.documents_sampling.component import documents_sampling
from kfp_components.components.data_processing.autorag.text_extraction.component import text_extraction

step_output_dir = Path("./step_outputs")
step_output_dir.mkdir(parents=True, exist_ok=True)

test_data_bucket_name = BUCKET_NAME
test_data_key = "benchmark.json"
input_data_bucket_name = BUCKET_NAME
input_data_key = ""
sampling_config = {}

#### Step 1: Test data loader

Downloads the benchmark (test data) JSON file from the configured S3 bucket and key. The file must be a JSON containing a list of items with `question`, `correct_answers`, and `correct_answer_document_ids`.

In [12]:
test_data_out = SimpleNamespace(path=str(step_output_dir / "test_data.json"))

test_data_loader.python_func(
    test_data_bucket_name=test_data_bucket_name,
    test_data_path=test_data_key,
    test_data=test_data_out,
)

output_path = Path(test_data_out.path)
with output_path.open("r", encoding="utf-8") as f:
    test_data = json.load(f)

print(json.dumps(test_data, indent=4, ensure_ascii=False))

Fetching test data from S3: bucket=autorag-dev-preview-dataset, path=benchmark.json
Starting download to step_outputs/test_data.json
Download completed successfully
[
    {
        "question": "What was IBM's revenue in the first quarter of 2024?",
        "correct_answers": [
            "Revenue of $14.5 billion, up 1 percent, up 3 percent at constant currency."
        ],
        "correct_answer_document_ids": [
            "ibm-1q25-earnings-press-release.pdf"
        ]
    },
    {
        "question": "What did IBM announce regarding HashiCorp in first quarter 2024?",
        "correct_answers": [
            "IBM announced its intent to acquire HashiCorp, Inc. for $35 per share in cash, representing an enterprise value of $6.4 billion. The transaction was expected to close by the end of 2024."
        ],
        "correct_answer_document_ids": [
            "ibm-1q25-earnings-press-release.pdf"
        ]
    }
]


#### Step 2: Documents sampling

Lists objects in the S3 input bucket, filters by supported extensions (e.g. `.pdf`, `.docx`, `.pptx`, `.md`, `.html`, `.txt`), and builds a sampled set: documents referenced in the benchmark (from Step 1) are prioritized; then others are added until a configurable size limit (1 GB as default) is reached. **This step does not download document contents.** It writes a YAML manifest, `sampled_documents_descriptor.yaml`, containing bucket, prefix, and the list of selected object keys and sizes. That manifest is the input for the text extraction step.

In [13]:
test_data_in = SimpleNamespace(path=str(step_output_dir / "test_data.json"))
sampled_documents_out = SimpleNamespace(path=str(step_output_dir / "sampled_documents"))

documents_sampling.python_func(
    input_data_bucket_name=input_data_bucket_name,
    input_data_path=input_data_key,
    test_data=test_data_in,
    sampling_config=sampling_config,
    sampled_documents=sampled_documents_out,
)

descriptor_path = step_output_dir / "sampled_documents" / "sampled_documents_descriptor.yaml"
with open(descriptor_path) as f:
    descriptor = yaml.safe_load(f)

print(json.dumps(descriptor, indent=4, ensure_ascii=False))

Sampled documents descriptor content {'bucket': 'autorag-dev-preview-dataset', 'prefix': '', 'documents': [{'key': 'ibm-1q25-earnings-press-release.pdf', 'size_bytes': 230841}, {'key': '4q25-press-release.pdf', 'size_bytes': 278516}, {'key': 'ibm-1q24-earnings-press-release.pdf', 'size_bytes': 211288}, {'key': 'ibm-2q24-earnings-press-release.pdf', 'size_bytes': 278765}, {'key': 'ibm-2q25-earnings-press-release.pdf', 'size_bytes': 280313}, {'key': 'ibm-3q-25-press-release.pdf', 'size_bytes': 275585}, {'key': 'ibm-3q24-earnings-press-release.pdf', 'size_bytes': 280378}, {'key': 'ibm-4q24-earnings-press-release.pdf', 'size_bytes': 290324}], 'total_size_bytes': 2126010, 'count': 8}
Sampled documents descriptor written to step_outputs/sampled_documents/sampled_documents_descriptor.yaml
{
    "bucket": "autorag-dev-preview-dataset",
    "prefix": "",
    "documents": [
        {
            "key": "ibm-1q25-earnings-press-release.pdf",
            "size_bytes": 230841
        },
        {
 

#### Step 3: Text extraction

Reads the `sampled_documents_descriptor.yaml` produced by Step 2, downloads each listed document from S3 into a temporary directory, and runs **Docling** to extract text. Output is one Markdown file per document (e.g. `document_0.md`, `document_1.md`) written to the artifact output path. These files are the final text corpus for the experiment.

In [14]:
sampled_descriptor_in = SimpleNamespace(path=str(step_output_dir / "sampled_documents"))
extracted_text_out = SimpleNamespace(path=str(step_output_dir / "extracted_text"))

text_extraction.python_func(
    sampled_documents_descriptor=sampled_descriptor_in,
    extracted_text=extracted_text_out,
)

Starting text extraction for 8 documents.
Processing document: ibm-2q24-earnings-press-release.pdf
Processing document: ibm-3q24-earnings-press-release.pdf
Processing document: ibm-4q24-earnings-press-release.pdf
Processing document: ibm-2q25-earnings-press-release.pdf
Processing document: ibm-3q-25-press-release.pdf
Processing document: ibm-1q25-earnings-press-release.pdf
Processing document: 4q25-press-release.pdf
Processing document: ibm-1q24-earnings-press-release.pdf
Successfully extracted text from ibm-1q25-earnings-press-release.pdf
Successfully extracted text from ibm-1q24-earnings-press-release.pdf
Successfully extracted text from ibm-2q25-earnings-press-release.pdf
Successfully extracted text from ibm-2q24-earnings-press-release.pdf
Successfully extracted text from ibm-3q24-earnings-press-release.pdf
Successfully extracted text from ibm-3q-25-press-release.pdf
Successfully extracted text from 4q25-press-release.pdf
Successfully extracted text from ibm-4q24-earnings-press-rele

Load the extracted Markdown files from Step 3 into LangChain `Document` objects.
> 🔖 **Note:** Document metadata must contain `document_id` key with the name of the document as it was refered in the benchmark data json file.

In [15]:
paths = list(Path("step_outputs/extracted_text").glob("*.md"))
documents = [
    Document(
        page_content=p.read_text(encoding="utf-8", errors="replace"),
        metadata={"document_id": p.stem},
    )
    for p in sorted(paths)
]
documents[0].page_content[:500]

n = 3
print(f"First {n} documents:")
for doc in documents[:n]:
    print("=" * 100)
    print(doc.metadata)
    print(doc.page_content[:800])


First 3 documents:
{'document_id': '4q25-press-release'}
## IBM RELEASES FOURTH-QUARTER RESULTS

Strong, broad-based performance, led by double-digit Software and Infrastructure growth; Double-digit growth in full-year profit and free cash flow

ARMONK, N.Y., January 28, 2026 . . . IBM (NYSE: IBM) today announced fourth-quarter 2025 earnings results.

'In the fourth quarter, we delivered strong revenue growth, with double -digit Software performance. Additionally, Infrastructure continued its double-digit revenue growth with the robust adoption of the next generation of our mainframe platform. Our generative AI book of business now stands at more than $12.5 billion. This capped a strong 2025 for IBM where we exceeded expectations for revenue, profit and free cash flow," said Arvind Krishna, IBM chairman, president and chief executive officer. "W
{'document_id': 'ibm-1q24-earnings-press-release'}
## IBM RELEASES FIRST-QUARTER RESULTS

## Accelerated Software revenue growth; Strong gross

## Run ai4rag experiment

## Summary and next steps

**Summary:** This notebook set up the experiment data, processed it, ran the ai4rag experiment, displayed the RAG patterns leaderboard, and queried the selected pattern.

**Next steps:**
